## GROUNDEDNESS
Groundedness evaluates how well the model's generated answers align with information from the input source. Even if the responses from LLM are factually correct, they will be considered ungrounded if they cannot be verified against the provided sources (such as your input source or your database).


In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load a sentence transformer model for embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Example reference text retrieved from a RAG system
retrieved_text = "Paris is the capital of France. It is known for the Eiffel Tower."

# LLM-generated response
llm_response = "The capital of France is paris."

# Compute embeddings
retrieved_embedding = model.encode(retrieved_text)
response_embedding = model.encode(llm_response)

# Compute cosine similarity
similarity_score = cosine_similarity([response_embedding], [retrieved_embedding])[0][0]

print(f"Groundedness Score (0-1): {similarity_score:.2f}")


Groundedness Score (0-1): 0.80


## Coherence
Analyzing logical consistency and clarity over longer stretches of text

- coh-metrix

In [9]:
pip install textstat spacy

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 105 kB 5.8 MB/s eta 0:00:01
     |████████████████████████████████| 6.3 MB 12.1 MB/s eta 0:00:01
     |████████████████████████████████| 939 kB 19.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.1 MB 25.3 MB/s eta 0:00:01
     |████████████████████████████████| 45 kB 14.2 MB/s eta 0:00:01
     |████████████████████████████████| 182 kB 53.7 MB/s eta 0:00:01
     |████████████████████████████████| 50 kB 17.4 MB/s eta 0:00:01
     |████████████████████████████████| 431 kB 11.1 MB/s eta 0:00:01
     |████████████████████████████████| 129 kB 28.2 MB/s eta 0:00:01
     |████████████████████████████████| 42 kB 4.2 MB/s  eta 0:00:01
     |████████████████████████████████| 635 kB 15.7 MB/s eta 0:00:01
     |████████████████████████████████| 780 kB 13.2 MB/s eta 0:00:01
     |████████████████████████████████| 5.4 MB 17.7 MB/s eta 0:00:01
     |███████████████████████

In [11]:
import spacy
spacy.cli.download("en_core_web_sm")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [12]:
import textstat
import spacy

# Load Spacy model for NLP processing
nlp = spacy.load("en_core_web_sm")

# Sample text
text = """Artificial Intelligence (AI) is transforming industries by automating tasks. 
It is widely used in healthcare, finance, and education. AI systems learn from data 
to improve decision-making."""

# Compute readability scores
flesch_reading_ease = textstat.flesch_reading_ease(text)
flesch_kincaid_grade = textstat.flesch_kincaid_grade(text)
lexical_diversity = len(set(text.split())) / len(text.split())

# Compute syntactic complexity using Spacy
doc = nlp(text)
num_sentences = len(list(doc.sents))
num_words = len(text.split())
avg_sentence_length = num_words / num_sentences

# Display results
print(f"Flesch Reading Ease: {flesch_reading_ease:.2f}")
print(f"Flesch-Kincaid Grade Level: {flesch_kincaid_grade:.2f}")
print(f"Lexical Diversity: {lexical_diversity:.2f}")
print(f"Average Sentence Length: {avg_sentence_length:.2f}")


Flesch Reading Ease: 28.80
Flesch-Kincaid Grade Level: 11.40
Lexical Diversity: 0.96
Average Sentence Length: 8.67


- Higher Flesch Reading Ease = easier text
- Higher Lexical Diversity = richer vocabulary
- Longer sentences = more complex text

## Perplexity
Assessing the natural flow and readability of text generated by the LLM.Lower perplexity 2 values indicate better performance.Simply put, it quantifies how well the model predicted probability distribution aligns with the actual distribution of the words in the dataset.

Purpose:
 - Confidence of model predictions

In [15]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def calculate_perplexity(text, model_name="gpt2"):
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name)
    model.eval()

    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    
    
    loss = outputs.loss
    perplexity = torch.exp(loss)
    return perplexity.item()

text = "The quick brown fox jumps over the lazy dog."
print("Perplexity:", calculate_perplexity(text))


Perplexity: 162.47021484375


General Perplexity Ranges

    Very Low (1 - 10) → Excellent prediction
        The model is highly confident and generates nearly deterministic predictions.
        Typically seen in models trained on highly structured or repetitive datasets.

    Moderate (10 - 100) → Good prediction
        The model performs well but has some uncertainty.
        Common for well-trained language models on structured text data.

    High (100 - 1000) → Average performance
        The model struggles with prediction accuracy.
        Often seen when the model is undertrained or when there’s domain mismatch.

    Very High (>1000) → Poor prediction
        The model is highly uncertain and likely producing random or incorrect outputs.
        Can happen if the model is untrained or dealing with out-of-distribution data.

## BLEU (Bilingual Evaluation Understudy)
BLEU is a metric commonly used in machine translation tasks. It compares the generated output with one or more reference translations and measures the similarity between them.

In [17]:
pip install nltk

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.5 MB 6.2 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [18]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Reference sentences (human translations)
reference = [
    "The cat is on the mat.",
    "A cat is sitting on the mat."
]

# Candidate sentence (machine-generated translation)
candidate = "The cat is sitting on the mat."

# Tokenize sentences
reference_tokens = [ref.split() for ref in reference]  # List of reference tokenized sentences
candidate_tokens = candidate.split()  # Tokenized candidate sentence

# Compute BLEU score
bleu_score = sentence_bleu(reference_tokens, candidate_tokens, 
                           smoothing_function=SmoothingFunction().method1)  # Smooth to handle zero scores

print(f"BLEU Score: {bleu_score:.4f}")


BLEU Score: 0.9306


# BLEU Score with Different n-gram Weights (BLEU-1, BLEU-2, BLEU-4)

In [19]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Reference sentences (human translations)
reference = [
    "The cat is on the mat.",
    "A cat is sitting on the mat."
]

# Candidate sentence (machine-generated translation)
candidate = "The cat is sitting on the mat."

# Tokenize sentences
reference_tokens = [ref.split() for ref in reference]  # List of reference tokenized sentences
candidate_tokens = candidate.split()  # Tokenized candidate sentence

# Compute BLEU scores with different n-gram weights
bleu_1 = sentence_bleu(reference_tokens, candidate_tokens, 
                        weights=(1.0, 0, 0, 0), 
                        smoothing_function=SmoothingFunction().method1)

bleu_2 = sentence_bleu(reference_tokens, candidate_tokens, 
                        weights=(0.5, 0.5, 0, 0), 
                        smoothing_function=SmoothingFunction().method1)

bleu_4 = sentence_bleu(reference_tokens, candidate_tokens, 
                        weights=(0.25, 0.25, 0.25, 0.25), 
                        smoothing_function=SmoothingFunction().method1)

# Print results
print(f"BLEU-1 (Unigram): {bleu_1:.4f}")
print(f"BLEU-2 (Bigram): {bleu_2:.4f}")
print(f"BLEU-4 (Full BLEU): {bleu_4:.4f}")


BLEU-1 (Unigram): 1.0000
BLEU-2 (Bigram): 1.0000
BLEU-4 (Full BLEU): 0.9306
